# 02 — Trajectory Collection

Collect **1000 diverse trajectories** to train the preference reward model:
- **600 expert** episodes from the trained baseline PPO (stochastic rollouts)
- **400 random** episodes from a purely random policy

This diversity is critical — if all trajectories are expert-level, preference pairs carry no signal and the reward model learns nothing. Mixing in random/bad episodes gives the model clear examples of what *not* to do.

In [1]:
import sys
sys.path.insert(0, '../src')

import pickle
import numpy as np
import gymnasium as gym
from stable_baselines3 import PPO
from pathlib import Path

CHECKPOINT_PATH = Path('../checkpoints/baseline_ppo.zip')
DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)
N_TRAJECTORIES = 1000

assert CHECKPOINT_PATH.exists(), f'Run notebook 01 first! Missing {CHECKPOINT_PATH}'
print('Setup complete.')

Setup complete.


In [2]:
# Load baseline PPO
env = gym.make('LunarLander-v3')
model = PPO.load(str(CHECKPOINT_PATH), env=env)
print('Model loaded.')

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Model loaded.


/home/har5ha/Desktop/LunarLander/.venv/lib/python3.11/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [ ]:
def collect_episodes(policy, env, n, deterministic=False, source_label='expert'):
    """Roll out `policy` for `n` episodes. Pass policy=None for random."""
    trajs = []
    for ep in range(n):
        obs, _ = env.reset()
        states, actions, rewards = [], [], []
        done = False
        landed = False
        while not done:
            if policy is None:
                action = env.action_space.sample()
            else:
                action, _ = policy.predict(obs, deterministic=deterministic)
                action = int(action)
            states.append(obs.copy())
            actions.append(int(action))
            obs, reward, terminated, truncated, _ = env.step(action)
            rewards.append(float(reward))
            done = terminated or truncated
            if terminated and reward == 100:
                landed = True
        trajs.append({
            'states': np.array(states, dtype=np.float32),
            'actions': np.array(actions, dtype=np.int32),
            'rewards': np.array(rewards, dtype=np.float32),
            'total_return': float(np.sum(rewards)),
            'landed': landed,
            'length': len(rewards),
            'source': source_label,
        })
        if (ep + 1) % 100 == 0:
            print(f'  {source_label}: {ep+1}/{n}')
    return trajs


env = gym.make('LunarLander-v3')
model = PPO.load(str(CHECKPOINT_PATH))

print('Collecting expert trajectories (baseline PPO, stochastic)...')
expert_trajs = collect_episodes(model, env, n=600, deterministic=False, source_label='expert')

print('Collecting random trajectories...')
random_trajs = collect_episodes(None, env, n=400, deterministic=False, source_label='random')

env.close()

import random as _random
trajectories = expert_trajs + random_trajs
_random.shuffle(trajectories)
print(f'\nTotal: {len(trajectories)} trajectories ({len(expert_trajs)} expert + {len(random_trajs)} random)')

In [ ]:
# Summary statistics broken down by source
import matplotlib.pyplot as plt

for source in ['expert', 'random']:
    subset = [t for t in trajectories if t['source'] == source]
    returns = [t['total_return'] for t in subset]
    landed = sum(t['landed'] for t in subset) / len(subset)
    print(f'{source:8s} | n={len(subset):4d} | '
          f'return: mean={np.mean(returns):7.1f} std={np.std(returns):6.1f} '
          f'min={np.min(returns):7.1f} max={np.max(returns):7.1f} | '
          f'landed={landed:.0%}')

all_returns = [t['total_return'] for t in trajectories]
print(f'\nOverall return range: {np.min(all_returns):.1f} → {np.max(all_returns):.1f}')
print(f'Return std (diversity): {np.std(all_returns):.1f}  (higher = better reward model signal)')

In [5]:
# Save trajectories
out_path = DATA_DIR / 'trajectories.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(trajectories, f)
print(f'Saved {len(trajectories)} trajectories to {out_path}')
print(f'File size: {out_path.stat().st_size / 1024:.1f} KB')

Saved 1000 trajectories to ../data/trajectories.pkl
File size: 11558.8 KB


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

expert_returns = [t['total_return'] for t in trajectories if t['source'] == 'expert']
random_returns  = [t['total_return'] for t in trajectories if t['source'] == 'random']

axes[0].hist(expert_returns, bins=30, color='steelblue', edgecolor='black', alpha=0.75, label='Expert (PPO)')
axes[0].hist(random_returns,  bins=30, color='salmon',    edgecolor='black', alpha=0.75, label='Random')
axes[0].set_xlabel('Total Return')
axes[0].set_ylabel('Count')
axes[0].set_title('Trajectory Return Distribution by Source')
axes[0].legend()

lengths = [t['length'] for t in trajectories]
axes[1].hist(lengths, bins=30, color='mediumpurple', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Episode Length (steps)')
axes[1].set_ylabel('Count')
axes[1].set_title('Episode Length Distribution')

plt.tight_layout()
plt.savefig('../checkpoints/trajectory_stats.png', dpi=100)
plt.show()